In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install transformers datasets seqeval accelerate gradio -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.8 MB/s eta 0:00:00


In [6]:
import os
import json
import copy
import logging
import shutil
import torch
import re
import gradio as gr
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset as HFDataset
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [7]:
# ===== 2. TẢI DỮ LIỆU TỪ DRIVE HOẶC GDOWN =====
!gdown 1DuwYRftQjQmQcAR4FVPH-HvGuxGi4ist
!gdown 11xZZfla8CDH54-EeUUdnAAoT2ummuEJh
!gdown 1wVyhQkhAzwod2at7Ir3tRHbdCMOszzwg


Downloading...
From: https://drive.google.com/uc?id=1DuwYRftQjQmQcAR4FVPH-HvGuxGi4ist
To: /content/train_word.conll
100% 1.42M/1.42M [00:00<00:00, 137MB/s]
Downloading...
From: https://drive.google.com/uc?id=11xZZfla8CDH54-EeUUdnAAoT2ummuEJh
To: /content/test_word.conll
100% 958k/958k [00:00<00:00, 144MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wVyhQkhAzwod2at7Ir3tRHbdCMOszzwg
To: /content/dev_word.conll
100% 628k/628k [00:00<00:00, 118MB/s]


In [8]:
# ===== 3. ĐỌC FILE CONLL VÀ XỬ LÝ DỮ LIỆU =====
def read_conll(file_path):
    sentences, sentence_labels, unique_labels = [], [], set()
    with open(file_path, 'r', encoding='utf-8') as file:
        tokens, labels = [], []
        for line in file:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(' '.join(tokens))
                    sentence_labels.append(' '.join(labels))
                    tokens, labels = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                label = parts[1] if len(parts) > 1 else 'O'
                labels.append(label)
                unique_labels.add(label)
        if tokens:
            sentences.append(' '.join(tokens))
            sentence_labels.append(' '.join(labels))
    print(f"Unique labels found: {unique_labels}")
    return sentences, sentence_labels, sorted(unique_labels)

train_sentences, train_labels, labels_set = read_conll("./train_word.conll")
dev_sentences, dev_labels, _ = read_conll("./dev_word.conll")
test_sentences, test_labels, _ = read_conll("./test_word.conll")



Unique labels found: {'B-NAME', 'B-ORGANIZATION', 'I-SYMPTOM_AND_DISEASE', 'I-PATIENT_ID', 'I-DATE', 'B-GENDER', 'B-TRANSPORTATION', 'I-TRANSPORTATION', 'B-SYMPTOM_AND_DISEASE', 'B-PATIENT_ID', 'B-AGE', 'I-AGE', 'I-JOB', 'I-NAME', 'B-JOB', 'I-ORGANIZATION', 'O', 'B-LOCATION', 'I-LOCATION', 'B-DATE'}
Unique labels found: {'B-NAME', 'B-ORGANIZATION', 'I-PATIENT_ID', 'I-DATE', 'B-GENDER', 'B-TRANSPORTATION', 'I-TRANSPORTATION', 'B-SYMPTOM_AND_DISEASE', 'B-PATIENT_ID', 'B-AGE', 'I-JOB', 'I-NAME', 'B-JOB', 'I-ORGANIZATION', 'B-DATE', 'O', 'B-LOCATION', 'I-LOCATION', 'I-SYMPTOM_AND_DISEASE'}
Unique labels found: {'B-NAME', 'B-ORGANIZATION', 'I-PATIENT_ID', 'I-DATE', 'B-GENDER', 'B-TRANSPORTATION', 'I-TRANSPORTATION', 'B-SYMPTOM_AND_DISEASE', 'B-PATIENT_ID', 'B-AGE', 'I-AGE', 'I-JOB', 'I-NAME', 'B-JOB', 'I-ORGANIZATION', 'B-DATE', 'O', 'B-LOCATION', 'I-LOCATION', 'I-SYMPTOM_AND_DISEASE'}


In [9]:
label_list = sorted(labels_set)
label_map = {label: i for i, label in enumerate(label_list)}

In [10]:
print("Label list khi train:")
for idx, label in enumerate(label_list):
    print(f"{idx}: {label}")

# Lưu ra file để dùng lại khi inference
with open("labels.txt", "w", encoding="utf-8") as f:
    for label in label_list:
        f.write(label + "\n")

Label list khi train:
0: B-AGE
1: B-DATE
2: B-GENDER
3: B-JOB
4: B-LOCATION
5: B-NAME
6: B-ORGANIZATION
7: B-PATIENT_ID
8: B-SYMPTOM_AND_DISEASE
9: B-TRANSPORTATION
10: I-AGE
11: I-DATE
12: I-JOB
13: I-LOCATION
14: I-NAME
15: I-ORGANIZATION
16: I-PATIENT_ID
17: I-SYMPTOM_AND_DISEASE
18: I-TRANSPORTATION
19: O


In [11]:
# ===== 4. TIỀN XỬ LÝ & CHUYỂN ĐỔI DỮ LIỆU =====
def prepare_dataset(sentences, labels):
    return {'tokens': sentences, 'labels': labels}

def process_string_to_array(dataset):
    return {
        'tokens': [s.split() for s in dataset['tokens']],
        'labels': [l.split() for l in dataset['labels']]
    }

train_data = process_string_to_array(prepare_dataset(train_sentences, train_labels))
dev_data = process_string_to_array(prepare_dataset(dev_sentences, dev_labels))
test_data = process_string_to_array(prepare_dataset(test_sentences, test_labels))

train_dataset = HFDataset.from_dict(train_data)
dev_dataset = HFDataset.from_dict(dev_data)
test_dataset = HFDataset.from_dict(test_data)

In [12]:

print(f"Train dataset size: {len(train_dataset)}")
print("Train dataset sample:", train_dataset[0])
print(f"Dev dataset size: {len(dev_dataset)}")
print("Dev dataset sample:", dev_dataset[0])
print(f"Test dataset size: {len(test_dataset)}")
print("Test dataset sample:", test_dataset[0])


Train dataset size: 5027
Train dataset sample: {'tokens': ['Đồng_thời', ',', 'bệnh_viện', 'tiếp_tục', 'thực_hiện', 'các', 'biện_pháp', 'phòng_chống', 'dịch_bệnh', 'COVID', '-', '19', 'theo', 'hướng_dẫn', 'của', 'Bộ', 'Y_tế', '.'], 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'O']}
Dev dataset size: 2000
Dev dataset sample: {'tokens': ['Bác_sĩ', 'Nguyễn_Trung_Nguyên', ',', 'Giám_đốc', 'Trung_tâm', 'Chống', 'độc', ',', 'Bệnh_viện', 'Bạch_Mai', ',', 'cho', 'biết', 'bệnh_nhân', 'được', 'chuyển', 'đến', 'bệnh_viện', 'ngày', '7/3', ',', 'chẩn_đoán', 'ngộ_độc', 'thuốc', 'điều_trị', 'sốt_rét', 'chloroquine', '.'], 'labels': ['O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DATE', 'O', 'O', 'B-SYMPTOM_AND_DISEASE', 'I-SYMPTOM_AND_DISEASE', 'O', 'O', 'O', 'O']}
Test dataset size: 3000
Test datas

In [13]:
# ===== 5. CHUYỂN DỮ LIỆU SANG OBJECTS =====
class Example:
    def __init__(self, words, slot_labels, guid=None):
        self.words = words
        self.slot_labels = slot_labels
        self.guid = guid

def convert_dataset_to_examples(dataset):
    return [Example(words=tokens, slot_labels=labels, guid=i)
            for i, (tokens, labels) in enumerate(zip(dataset['tokens'], dataset['labels']))]

train_examples = convert_dataset_to_examples(train_dataset)
dev_examples = convert_dataset_to_examples(dev_dataset)
test_examples = convert_dataset_to_examples(test_dataset)


In [14]:
# ===== 6. CHUYỂN SANG FEATURE INPUT CHO MODEL =====
class InputFeatures:
    def __init__(self, input_ids, attention_mask, token_type_ids, slot_labels_ids):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.token_type_ids = token_type_ids
        self.slot_labels_ids = slot_labels_ids


In [15]:

def convert_examples_to_features(
    examples, max_seq_len, tokenizer, label_map, pad_label_id=-100
):
    features = []
    for example in examples:
        tokens, label_ids = [], []
        for word, label in zip(example.words, example.slot_labels):
            word_tokens = tokenizer.tokenize(word) or [tokenizer.unk_token]
            tokens.extend(word_tokens)
            label_ids.extend([label_map[label]] + [pad_label_id] * (len(word_tokens) - 1))
        special_tokens_count = 2
        if len(tokens) > max_seq_len - special_tokens_count:
            tokens = tokens[:max_seq_len - special_tokens_count]
            label_ids = label_ids[:max_seq_len - special_tokens_count]
        tokens = [tokenizer.cls_token] + tokens + [tokenizer.sep_token]
        label_ids = [pad_label_id] + label_ids + [pad_label_id]
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)
        token_type_ids = [0] * len(input_ids)  # RoBERTa chỉ có segment_id=0
        padding_length = max_seq_len - len(input_ids)
        input_ids += [tokenizer.pad_token_id] * padding_length
        attention_mask += [0] * padding_length
        token_type_ids += [0] * padding_length
        label_ids += [pad_label_id] * padding_length
        features.append(InputFeatures(input_ids, attention_mask, token_type_ids, label_ids))
    return features


In [16]:

# ===== 7. TOKENIZER & FEATURIZATION =====
from transformers import RobertaTokenizerFast

tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base', add_prefix_space=True)
max_seq_len = 128

train_features = convert_examples_to_features(train_examples, max_seq_len, tokenizer, label_map)
dev_features = convert_examples_to_features(dev_examples, max_seq_len, tokenizer, label_map)
test_features = convert_examples_to_features(test_examples, max_seq_len, tokenizer, label_map)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [17]:

# ===== 8. ĐÓNG GÓI DỮ LIỆU DẠNG PYTORCH DATASET =====
class NERDataset(Dataset):
    def __init__(self, features):
        self.features = features
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        feature = self.features[idx]
        return {
            'input_ids': torch.tensor(feature.input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(feature.attention_mask, dtype=torch.long),
            'token_type_ids': torch.tensor(feature.token_type_ids, dtype=torch.long),
            'labels': torch.tensor(feature.slot_labels_ids, dtype=torch.long),
        }

train_dataset = NERDataset(train_features)
dev_dataset = NERDataset(dev_features)
test_dataset = NERDataset(test_features)

In [18]:
# ===== 9. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH RoBERTa CHO NER =====
# Tự động đăng nhập wandb
os.environ["WANDB_API_KEY"] = "fb80f73fcb020dd331c4509a5851a96be928f490"  # Thêm mã API của bạn ở đây

# Load model với số lượng nhãn bằng label_list
model = AutoModelForTokenClassification.from_pretrained("roberta-base",num_labels=len(label_list))

# Hàm tính metrics
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    labels = p.label_ids
    true_preds, true_labels = [], []

    for pred, label in zip(preds, labels):
        tmp_preds, tmp_labels = [], []
        for p_i, l_i in zip(pred, label):
            if l_i != -100:
                tmp_preds.append(label_list[p_i])
                tmp_labels.append(label_list[l_i])
        true_preds.append(tmp_preds)
        true_labels.append(tmp_labels)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


training_args = TrainingArguments(
    output_dir="./ner_roberta",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    report_to=["wandb"],  # Không cần wandb, hoặc có thì sửa lại API key
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Huấn luyện model
trainer.train()

# Đánh giá trên dev
eval_metrics = trainer.evaluate()
print("==== ĐÁNH GIÁ TRÊN DEV ====")
print(eval_metrics)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-18-21cd60c29620>:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: vkhlinh (vkhlinh-l) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,1.364700
20,0.838200
30,0.653500
40,0.515000
50,0.510700
60,0.403100
70,0.388900
80,0.291200
90,0.306500
100,0.270600


==== ĐÁNH GIÁ TRÊN DEV ====
{'eval_loss': 0.12651993334293365, 'eval_precision': 0.8918173460006129, 'eval_recall': 0.9112259276655706, 'eval_f1': 0.9014171764888096, 'eval_runtime': 14.3872, 'eval_samples_per_second': 139.012, 'eval_steps_per_second': 8.688, 'epoch': 3.0}


In [19]:
# Đánh giá trên test
test_metrics = trainer.evaluate(test_dataset)
print("==== ĐÁNH GIÁ TRÊN TEST ====")
print(f"Loss: {test_metrics['eval_loss']:.4f}")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall: {test_metrics['eval_recall']:.4f}")
print(f"F1 Score: {test_metrics['eval_f1']:.4f}")


==== ĐÁNH GIÁ TRÊN TEST ====
Loss: 0.1513
Precision: 0.8807
Recall: 0.9016
F1 Score: 0.8910


In [20]:
# Xuất báo cáo chi tiết
predictions, labels, _ = trainer.predict(test_dataset)
preds = predictions.argmax(-1)
true_labels = []
true_preds = []
for pred, label in zip(preds, labels):
    temp_true = []
    temp_pred = []
    for p_i, l_i in zip(pred, label):
        if l_i != -100:
            temp_true.append(label_list[l_i])
            temp_pred.append(label_list[p_i])
    true_labels.append(temp_true)
    true_preds.append(temp_pred)

print("\n====== Báo cáo chi tiết theo từng thực thể ======")
print(classification_report(true_labels, true_preds, digits=4))




====== Báo cáo chi tiết theo từng thực thể ======
                     precision    recall  f1-score   support

                AGE     0.9231    0.9730    0.9474       518
               DATE     0.9756    0.9883    0.9820      1459
             GENDER     0.8844    0.9630    0.9220       405
                JOB     0.4118    0.3161    0.3577       155
           LOCATION     0.8735    0.9047    0.8888      3671
               NAME     0.8608    0.7907    0.8242       258
       ORGANIZATION     0.7554    0.7864    0.7706       707
         PATIENT_ID     0.9611    0.9802    0.9706      1665
SYMPTOM_AND_DISEASE     0.7388    0.7566    0.7476       916
     TRANSPORTATION     0.9518    0.9186    0.9349       172

          micro avg     0.8807    0.9016    0.8910      9926
          macro avg     0.8336    0.8378    0.8346      9926
       weighted avg     0.8792    0.9016    0.8900      9926



In [21]:
# Lưu model & tokenizer
save_directory = '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned'
os.makedirs(save_directory, exist_ok=True)
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

('/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned/vocab.json',
 '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned/merges.txt',
 '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned/added_tokens.json',
 '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned/tokenizer.json')

In [22]:
# ===== 10. INFERENCE - DỰ ĐOÁN THỰC THỂ =====

# Nạp lại model đã lưu
model_dir = '/content/drive/MyDrive/Ner_RoBERTa_finetuned/roberta-ner-finetuned'
model = AutoModelForTokenClassification.from_pretrained(model_dir)
tokenizer = RobertaTokenizerFast.from_pretrained(model_dir)

id2label = {i: label for i, label in enumerate(label_list)}

def split_sentences(text):
    # Tách câu theo dấu câu, giữ lại dấu câu ở cuối
    sentences = re.split(r'(?<=[.!?…])\s+', text.strip())
    # Loại bỏ câu rỗng
    sentences = [s for s in sentences if s.strip()]
    return sentences

def predict_entities_for_text(text):
    sentences = split_sentences(text)
    print(f"Phát hiện {len(sentences)} câu trong đoạn văn.")
    for idx, sent in enumerate(sentences, 1):
        print(f"\n------ Câu {idx} ------")
        # Tokenize
        inputs = tokenizer(sent, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1)
        predicted_labels = [id2label[label.item()] for label in predictions[0]]
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        print(f"{'Token':<15} {'Predicted Label'}")
        print("-" * 40)
        for token, label in zip(tokens, predicted_labels):
            print(f"{token:<15} {label}")

# Dùng thử
text = input("Nhập đoạn văn để nhận dạng thực thể: ")
predict_entities_for_text(text)


Nhập đoạn văn để nhận dạng thực thể: Hôm nay, ngày 8 tháng 6 năm 2025, Nguyễn Văn An, một kỹ sư phần mềm, đã tham gia hội thảo công nghệ tại Hà Nội. Anh ấy làm việc cho công ty Công nghệ FPT, một trong những tập đoàn lớn nhất Việt Nam. Hội thảo được tổ chức tại Trung tâm Hội nghị Quốc gia, nơi thu hút hơn 500 chuyên gia từ khắp nơi. An đã gặp Trần Thị Bình, một nhà nghiên cứu AI đến từ Đại học Bách Khoa. Họ cùng thảo luận về dự án trí tuệ nhân tạo với Google
Phát hiện 5 câu trong đoạn văn.

------ Câu 1 ------
Token           Predicted Label
----------------------------------------
<s>             O
ĠH              O
Ã´              O
m               O
Ġn              O
ay              O
,               O
Ġng             O
Ãł              O
y               O
Ġ8              B-DATE
Ġth             B-DATE
Ã¡              B-DATE
ng              I-DATE
Ġ6              I-DATE
Ġn              O
Ä               O
ĥ               O
m               O
Ġ2025           O
,               O
ĠN      

In [27]:
def predict_entities_word_level(text):
    sentences = split_sentences(text)
    print(f"Phát hiện {len(sentences)} câu trong đoạn văn.")
    for idx, sent in enumerate(sentences, 1):
        print(f"\n------ Câu {idx} ------")
        words = sent.split()
        encoding = tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        )
        # Remove the 'offset_mapping' key as the model's forward method doesn't accept it.
        # Alternatively, you could explicitly pass only the expected arguments:
        # outputs = model(input_ids=encoding['input_ids'], attention_mask=encoding['attention_mask'], token_type_ids=encoding['token_type_ids'])
        if 'offset_mapping' in encoding:
            encoding.pop('offset_mapping')

        with torch.no_grad():
            outputs = model(**encoding)
            logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)[0].tolist()  # [0]: batch size 1
        word_ids = encoding.word_ids(0)  # word_ids của câu đầu tiên
        word_labels = []
        for i, word in enumerate(words):
            # Tìm vị trí sub-token đầu tiên thuộc về từ i
            try:
                token_idx = word_ids.index(i)
                label_idx = predictions[token_idx]
                label = id2label[label_idx]
            except ValueError:
                # Không tìm thấy (từ bị cắt quá dài...)
                label = 'O'
            word_labels.append((word, label))

        print(f"{'Word':<20} {'Predicted Label'}")
        print("-" * 40)
        for word, label in word_labels:
            print(f"{word:<20} {label}")

# Dùng thử
text = input("Nhập đoạn văn để nhận dạng thực thể: ")
predict_entities_word_level(text)

Nhập đoạn văn để nhận dạng thực thể: Hôm nay, ngày 8 tháng 6 năm 2025, Nguyễn Văn An, một kỹ sư phần mềm, đã tham gia hội thảo công nghệ tại Hà Nội. Anh ấy làm việc cho công ty Công nghệ FPT, một trong những tập đoàn lớn nhất Việt Nam. Hội thảo được tổ chức tại Trung tâm Hội nghị Quốc gia, nơi thu hút hơn 500 chuyên gia từ khắp nơi. An đã gặp Trần Thị Bình, một nhà nghiên cứu AI đến từ Đại học Bách Khoa. Họ cùng thảo luận về dự án trí tuệ nhân tạo với Google.
Phát hiện 5 câu trong đoạn văn.

------ Câu 1 ------
Word                 Predicted Label
----------------------------------------
Hôm                  O
nay,                 O
ngày                 O
8                    B-DATE
tháng                B-DATE
6                    I-DATE
năm                  O
2025,                O
Nguyễn               B-NAME
Văn                  B-LOCATION
An,                  I-LOCATION
một                  O
kỹ                   O
sư                   O
phần                 O
mềm,                 O